# P04 · Notebook 15 · Do the newest machine-sound models beat BEATs in factory noise?

**Why:** every sound result so far uses BEATs (2022), frozen. Two newer models were built for machine signals, and both are open (MIT):
- **FISHER** (IEEE TII 2026);
- **ECHO** (ICASSP 2026).

Both split the spectrum into frequency bands and model each band. Factory noise is not spread evenly over frequency, so they might hold up better in it.

**What runs:**
- pumps and fans, at 6 dB and −6 dB;
- one microphone (CH0), and the 8 microphones aligned with the installation delays (BF);
- each new model frozen, with the same nearest-neighbour scorer and the same splits as notebooks 11 and 13;
- BEATs comes from those notebooks' files in Drive, so it is not recomputed.

**Decisions:** the better new model is chosen on pumps. The decisions are then made on fans, which play no part in the choice. Everything is fixed in `results/mimii_beats/RULES_new_encoders_written_before_running.md`.

**Run:**
- Runtime › Change runtime type › T4 GPU, then Runtime › Run all.
- Cell 4 loads both models and tests them on a synthetic clip **before** any download. If a model fails there, the run goes on with the other one.
- It downloads four zips of 8–11 GB, and saves each result to Drive › `P04_Encoders` as soon as it is done. If Colab disconnects, run all again: finished parts are skipped.
- It needs notebooks 11 and 13's files in Drive (`P04_ArrayNoise`, `P04_ArrayNoise_Fan`).

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, importlib
from pathlib import Path
import numpy as np
import torch
import torchaudio
from sklearn.metrics import roc_auc_score

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "omegaconf", "timm", "safetensors", "huggingface_hub"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK = MY / "P04_Encoders", Path("/content/work")
    ARR = {"pump": MY / "P04_ArrayNoise", "fan": MY / "P04_ArrayNoise_Fan"}
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_Encoders")), Path(os.environ.get("P04_WORK", "work"))
    ARR = {"pump": Path(os.environ.get("P04_ARR_PUMP", "P04_ArrayNoise")), "fan": Path(os.environ.get("P04_ARR_FAN", "P04_ArrayNoise_Fan"))}
for p in (OUT, WORK):
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: this will be very slow)", "· torch", torch.__version__)


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()

In [ ]:
# 2 · Every number below was fixed in the rules before any FISHER or ECHO embedding existed
RULES = "results/mimii_beats/RULES_new_encoders_written_before_running.md"
Z = "https://zenodo.org/records/3384388/files/"
LEVELS = {"pump|6dB": (Z + "6_dB_pump.zip?download=1", "6_dB_pump.zip", "a09ba6060c10fc09cd4c8770213b0b9f"),
          "pump|-6dB": (Z + "-6_dB_pump.zip?download=1", "-6_dB_pump.zip", "d20b783a0ff9c93d58f452f98c37b112"),
          "fan|6dB": (Z + "6_dB_fan.zip?download=1", "6_dB_fan.zip", "0890f7d3c2fd8448634e69ff1d66dd47"),
          "fan|-6dB": (Z + "-6_dB_fan.zip?download=1", "-6_dB_fan.zip", "f02ae808a58d84b6815b7ec38ff30879")}
if TEST:
    _t = json.loads(os.environ["P04_TEST_DATA"])
    LEVELS = {f"{m}|{lv}": tuple(_t[lv]) for m in ("pump", "fan") for lv in ("6dB", "-6dB")}
BEATS_FILE = {"pump": "array_{lv}_pump.npz", "fan": "fan_{lv}.npz"}        # notebooks 11 and 13
NFFT, HOP, UPS, MAXLAG = 1024, 512, 16, 16                                # notebook 11's GCC-PHAT (smoke test of cell 3)
SR = 16000
FRONTS = ["CH0", "BF"]
SHOTS = [1, 3, 10]
SPLITS = ["A", "B", "B+1", "B+3", "B+10"]
G0_REF = {"pump": {"CH0": 0.8234, "BF": 0.8795}, "fan": {"CH0": 0.7255, "BF": 0.8200}}   # −6 dB, Split A, B1
G1_MIN = 0.80                                                             # pump CH0 Split A at 6 dB
BEATS_FAN = {"A": 0.8200, "B+10": 0.5898}                                 # notebook 13, −6 dB, BF
BAR, BAR_ARRAY = 0.02, 0.01
FISHER_HF, ECHO_HF = "jiangab/FISHER-small-0723", "yucongzh/echo-small-0824"
FISHER_GIT = ("https://github.com/jianganbai/FISHER.git", "60dd0c8392b06a1e5ce337a92eb6785070fc1e06", {"models/fisher.py": "3e0c819c5ceab8d0e873776924ec0090ef36488a51094cc53c2f33c0e90e5cf3", "models/base.py": "5a74ee173fd4867947aca28f2aa5229eb013945a0d146e3c269f203f3656daa7", "models/images.py": "ca94a99be1aa9a7b1463db70397b9804c8b4718a9f869aa197f7a864d770d15c", "models/mae.py": "ceef8f11529cbe4f08b1ad36d2b6a7b63cdb62ce97e38fe252cab90df53e99ae", "models/modules.py": "084e7513bf5b71dfb140191c768b2acb5a94e185b8adda3a56230deb31590b14"})
EXPECT_DIM = {"FISHER": 4 * 384, "ECHO": 7 * 384}                         # at 16 kHz, as the rules state


def get_zip(level):
    url, name, want = LEVELS[level]
    for place in (OUT / name, WORK / name):
        if place.exists() and md5(place) == want:
            return place
    dest = WORK / name
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}")
    assert got == want, f"stop: {name} is not the published file"
    return dest


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def parse(names):
    pump = np.array([re.search(r"(id_\d+)", n).group(1) for n in names])
    label = np.array([1 if n.replace("\\", "/").split("/")[-2] == "abnormal" else 0 for n in names])
    return pump, label


def split_a(pump, label, fname, p):
    # the same split as notebooks 1 and 3: test = first N normal by name + all abnormal of pump p; train = the other normal
    idx = np.where(pump == p)[0]
    nor = idx[label[idx] == 0]; abn = idx[label[idx] == 1]
    nor = nor[np.argsort(fname[nor])]
    return nor[len(abn):], np.r_[nor[:len(abn)], abn]

In [ ]:
# 3 · Physics front end: GCC-PHAT delays from training clips, then delay-and-sum on the whole clip
WIN = np.hanning(NFFT).astype(np.float32)


def cross_spectra(x):
    # x: (samples, 8). Sum over frames of conj(X_0)·X_m for m = 0..7 → (8, NFFT/2+1) complex
    n = (len(x) - NFFT) // HOP + 1
    idx = np.arange(NFFT)[None, :] + HOP * np.arange(n)[:, None]
    F = np.fft.rfft(x[idx] * WIN[None, :, None], axis=1)           # (frames, bins, 8)
    return np.einsum("fb,fbm->mb", np.conj(F[:, :, 0]), F)


def delays_from(G):
    # GCC-PHAT on summed cross-spectra G (8, bins) → delay of each channel against channel 0, in samples
    out = np.zeros(G.shape[0])
    for m in range(1, G.shape[0]):
        g = G[m] / np.maximum(np.abs(G[m]), 1e-12)
        r = np.fft.irfft(g, n=NFFT * UPS)
        K = MAXLAG * UPS
        rc = np.concatenate([r[-K:], r[:K + 1]])                     # lags −K..+K (in 1/UPS samples)
        out[m] = (np.argmax(rc) - K) / UPS
    return out


def delay_and_sum(X, tau, N):
    k = np.arange(X.shape[0])[:, None]
    return np.fft.irfft((X * np.exp(2j * np.pi * k * tau[None, :] / N)).mean(1), n=N).astype(np.float32)


def front_ends(x, tau, tau_same=None):
    # x: (samples, 8) → mono signals; BF advances channel m by tau[m] samples, then averages
    N = len(x)
    X = np.fft.rfft(x, axis=0)
    out = {"CH0": x[:, 0].copy(), "AVG": x.mean(1).astype(np.float32), "BF": delay_and_sum(X, tau, N)}
    if tau_same is not None:
        out["BFs"] = delay_and_sum(X, tau_same, N)
    return out


# smoke test: a broadband source arriving with known fractional delays, plus independent noise 6 dB louder
rng = np.random.default_rng(0)
true = np.r_[0.0, rng.uniform(-4, 4, 7)]
s = rng.standard_normal(160000).astype(np.float32)
S = np.fft.rfft(s); kk = np.arange(len(S))
xs = np.stack([np.fft.irfft(S * np.exp(-2j * np.pi * kk * t / 160000), n=160000) for t in true], 1).astype(np.float32)
xn = xs + 2.0 * rng.standard_normal(xs.shape).astype(np.float32)
est = delays_from(cross_spectra(xn))
fe = front_ends(xn, est)
gain = lambda y: 10 * np.log10(np.var(s) / np.var(y - s * (np.dot(y, s) / np.dot(s, s))))
print("delays true", np.round(true, 2).tolist(), "\nestimated  ", np.round(est, 2).tolist())
print(f"SNR after each front end (dB): CH0 {gain(fe['CH0']):.1f} · AVG {gain(fe['AVG']):.1f} · BF {gain(fe['BF']):.1f}")
assert np.max(np.abs(est - true)) < 0.25, "smoke test: GCC-PHAT does not recover the delays"
assert gain(fe["BF"]) > gain(fe["CH0"]) + 6, "smoke test: delay-and-sum does not gain the expected ~9 dB"

In [ ]:
# 4 · The two new models: load each, then test it on a synthetic clip before any download
INFO = {}


def fisher_spec(w):
    # the FISHER README's input: log magnitude STFT (25 ms window, 10 ms hop, not centred), fixed normalisation, ≤ 1024 frames
    w = w - w.mean(dim=-1, keepdim=True)
    st = torchaudio.transforms.Spectrogram(n_fft=25 * SR // 1000, win_length=None, hop_length=10 * SR // 1000, power=1, center=False).to(w.device)
    s = torch.log(torch.abs(st(w)) + 1e-10).transpose(-2, -1)
    s = (s + 3.017344307886898) / (2.1531635155379805 * 2)
    return s[:, :1024].unsqueeze(1)                                       # (batch, 1, time, freq)


def same_weights(model, wfile):
    from safetensors.torch import load_file
    sd_file, sd_model = load_file(str(wfile)), model.state_dict()
    return all(k in sd_model and torch.equal(sd_model[k].cpu(), v) for k, v in sd_file.items())


def fisher_from_github(snap):
    # fallback: the authors' GitHub code at a checked commit, with the same Hugging Face weights
    url, commit, hashes = FISHER_GIT
    code = WORK / "FISHER_code"
    if not code.exists():
        subprocess.run(["git", "clone", "-q", url, str(code)], check=True)
    subprocess.run(["git", "-C", str(code), "checkout", "-q", commit], check=True)
    for f, h in hashes.items():
        assert sha256(code / f) == h, f"stop: {f} is not the checked FISHER code"
    for k in [k for k in sys.modules if k == "models" or k.startswith("models.")]:
        del sys.modules[k]
    sys.path.insert(0, str(code))
    from models.fisher import FISHER, Data2VecMultiConfig
    from safetensors.torch import load_file
    c = json.loads((snap / "config.json").read_text())
    cfg = Data2VecMultiConfig()
    for k in ("depth", "embed_dim", "num_heads", "band_width", "max_band_per_sample"):
        setattr(cfg, k, c[k])
    im = cfg.modalities.image
    im.embed_dim, im.band_width, im.num_alibi_heads, im.model_depth = c["embed_dim"], c["band_width"], c["num_heads"], c["depth"]
    m = FISHER(cfg)
    sd = {(k[6:] if k.startswith("model.") else k): v for k, v in load_file(str(snap / "model.safetensors")).items()}
    m.load_state_dict(sd, strict=True)
    INFO["FISHER"]["loader"] = f"GitHub code {commit[:7]} + Hugging Face weights"
    return m


def load_fisher():
    from huggingface_hub import snapshot_download
    snap = Path(snapshot_download(FISHER_HF))
    INFO["FISHER"] = {"repo": FISHER_HF, "revision": snap.name, "weights_sha256": sha256(snap / "model.safetensors")}
    try:
        from transformers import AutoModel
        m = AutoModel.from_pretrained(FISHER_HF, trust_remote_code=True)
        assert same_weights(m, snap / "model.safetensors"), "loaded weights differ from the file"
        INFO["FISHER"]["loader"] = "transformers AutoModel, as on the model card"
    except Exception as e:
        print("  FISHER via AutoModel failed:", repr(e)[:300], "→ the authors' GitHub code with the same weights")
        m = fisher_from_github(snap)
    return m.to(DEVICE).eval()


def load_echo():
    # as on the ECHO model card: its own code and model.safetensors from the same Hugging Face snapshot
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file
    snap = Path(snapshot_download(ECHO_HF))
    INFO["ECHO"] = {"repo": ECHO_HF, "revision": snap.name, "weights_sha256": sha256(snap / "model.safetensors"),
                    "code_sha256": sha256(snap / "audioMAE_band_upgrade.py")}
    saved = {k: sys.modules.pop(k) for k in ("modules", "utils") if k in sys.modules}
    sys.path.insert(0, str(snap))
    try:
        mod = importlib.import_module("audioMAE_band_upgrade")
    finally:
        sys.path.remove(str(snap))
        for k in ("modules", "utils"):
            sys.modules.pop(k, None)
        sys.modules.update(saved)
    m = mod.AudioMAEWithBand(spec_len=2000, band_width=32, shift_size=16, in_chans=1, embed_dim=384, encoder_depth=12,
                             num_heads=6, mlp_ratio=4.0, freq_pos_emb_dim=384)
    missing, unexpected = m.load_state_dict(load_file(str(snap / "model.safetensors")), strict=False)
    assert not missing, f"stop: ECHO weights missing for {list(missing)[:5]}"
    INFO["ECHO"].update(loader="model card: its own code + model.safetensors", unexpected_keys=len(unexpected))
    return m.to(DEVICE).eval()


def emb_fisher(m, waves):
    w = torch.from_numpy(np.stack(waves)).to(DEVICE)
    return m.extract_features(fisher_spec(w)).float().cpu().numpy()


def emb_echo(m, waves):
    out = []
    for x in waves:
        u, _ = m.extract_features_from_audio(torch.from_numpy(x)[None].to(DEVICE), sample_rate=SR)
        out.append(u.float().cpu().numpy())
    return np.stack(out)


class StandIn(torch.nn.Module):
    # TEST ONLY: a fixed random projection of log band energies, shaped like a band model's output. Never used on Colab.
    def __init__(self, bands, dim, seed):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.bands, self.P = bands, torch.randn(bands, 8, dim, generator=g)

    def _feat(self, spec):                                                # spec: (batch, time, freq)
        e = torch.stack([b.mean(-1) for b in spec.chunk(self.bands, dim=-1)], -1)          # (batch, time, bands)
        q = torch.stack([e[:, i * e.shape[1] // 8:(i + 1) * e.shape[1] // 8].mean(1) for i in range(8)], 1)   # (batch, 8, bands)
        return torch.einsum("bqk,kqd->bkd", q, self.P).flatten(1)

    def extract_features(self, spec):                                     # FISHER's interface
        return self._feat(spec[:, 0])

    def extract_features_from_audio(self, x, sample_rate):                # ECHO's interface
        return self._feat(fisher_spec(x)[:, 0])[0], None


LOADERS = {"FISHER": load_fisher, "ECHO": load_echo}
EMB = {"FISHER": emb_fisher, "ECHO": emb_echo}
if TEST:
    def _stand(name, bands, seed):
        INFO[name] = {"repo": "stand-in (TEST)", "loader": "stand-in", "weights_sha256": ""}
        return StandIn(bands, 16, seed)
    LOADERS = {"FISHER": lambda: _stand("FISHER", 4, 1), "ECHO": lambda: _stand("ECHO", 7, 2)}
    EXPECT_DIM = {"FISHER": 4 * 16, "ECHO": 7 * 16}

MODELS = {}
rng = np.random.default_rng(0)
probe = [(rng.standard_normal(10 * SR) * 0.1).astype(np.float32) for _ in range(3)]
for name, loader in LOADERS.items():
    try:
        t = time.time(); m = loader()
        e = EMB[name](m, probe); e1 = EMB[name](m, probe[:1])
        assert e.ndim == 2 and e.shape[0] == 3 and np.isfinite(e).all(), f"bad output {e.shape}"
        assert np.allclose(e[:1], e1, atol=1e-4), "a clip's embedding depends on the other clips in its batch"
        INFO[name].update(dim=int(e.shape[1]), load_s=round(time.time() - t, 1))
        note = "" if e.shape[1] == EXPECT_DIM[name] else f" · NOTE: the rules expected {EXPECT_DIM[name]}"
        MODELS[name] = m
        print(f"{name}: OK · {INFO[name]['loader']} · embedding of {e.shape[1]} numbers{note}")
    except Exception as ex:
        INFO.setdefault(name, {})["failed"] = repr(ex)[:500]
        print(f"{name}: FAILED, dropped from this run · {repr(ex)[:300]}")
assert MODELS, "stop: neither new model could be loaded"

In [ ]:
# 5 · The BEATs reference from Drive, the installation delays, and gate G0
def load_beats(m, lv):
    f = ARR[m] / BEATS_FILE[m].format(lv=lv)
    if not f.exists():
        raise SystemExit(f"missing {f}: run notebook {11 if m == 'pump' else 13} first")
    D = np.load(f, allow_pickle=False)
    return {"fname": D["fname"], "mid": D["pump"] if "pump" in D.files else D["mid"], "label": D["label"],
            "CH0": D["CH0_mean"].astype(np.float64), "BF": D["BF_mean"].astype(np.float64), "tau": json.loads(str(D["tau"]))}


BEATS = {key: load_beats(*key.split("|")) for key in LEVELS}
TAU = {m: {p: np.array(v) for p, v in BEATS[f"{m}|6dB"]["tau"].items()} for m in ("pump", "fan")}
for m in TAU:
    print(m, "installation delays (samples):", {p: np.round(v, 2).tolist() for p, v in TAU[m].items()})


def unit(E):
    return E / np.linalg.norm(E, axis=1, keepdims=True)


def b1(R, X):
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def b1_loo(R):
    S = unit(R) @ unit(R).T
    np.fill_diagonal(S, -np.inf)
    return 1.0 - S.max(1)


def evaluate(E, mid, label, names, fuse_with=None):
    # the splits of notebooks 11 and 13; with fuse_with, the score is the sum of both models' rescaled scores
    ids = sorted(set(mid.tolist()))
    A = {p: split_a(mid, label, names, p) for p in ids}
    r = {s: {} for s in SPLITS}
    for p in ids:
        tr, te = A[p]; y = label[te]
        fleet = np.concatenate([A[q][0] for q in ids if q != p])
        for s, ref in (("A", tr), ("B", fleet), *[(f"B+{n}", np.r_[fleet, tr[:n]]) for n in SHOTS]):
            if fuse_with is None:
                sc = b1(E[ref], E[te])
            else:
                sc = 0.0
                for F in (E, fuse_with):
                    l = b1_loo(F[ref]); sc = sc + (b1(F[ref], F[te]) - l.mean()) / l.std()
            r[s][p] = round(float(roc_auc_score(y, sc)), 4)
    for s in SPLITS:
        r[s]["avg"] = round(float(np.mean([r[s][p] for p in ids])), 4)
    return r


G0 = {}
for m in ("pump", "fan"):
    Bm = BEATS[f"{m}|-6dB"]
    for f in FRONTS:
        got = evaluate(Bm[f], Bm["mid"], Bm["label"], Bm["fname"])["A"]["avg"]
        G0[f"{m}|{f}"] = (got, G0_REF[m][f])
print("G0 · BEATs from Drive, −6 dB, Split A:", {k: v[0] for k, v in G0.items()})
if not TEST:
    bad = {k: v for k, v in G0.items() if v[0] != v[1]}
    if bad:
        raise SystemExit(f"stop: G0 failed, the Drive files do not give notebook 11/13's numbers: {bad}")
    print("G0 passed: the BEATs reference reproduces exactly")

In [ ]:
# 6 · For each machine and level: CH0 and BF through each new model, saved to Drive as it goes
def run(key, batch=16):
    out = OUT / f"enc_{key.replace('|', '_')}.npz"
    if out.exists():
        print(key, "already done:", out.name); return
    m, lv = key.split("|")
    zpath = get_zip(key); zf = zipfile.ZipFile(zpath)
    names = np.array(sorted(n for n in zf.namelist() if n.lower().endswith(".wav")))
    assert np.array_equal(names, BEATS[key]["fname"]), "stop: the clips differ from notebook 11/13's"
    mid, label = parse(names)
    E = {(n, f): [] for n in MODELS for f in FRONTS}; buf = {f: [] for f in FRONTS}
    secs = {n: 0.0 for n in MODELS}; t0 = time.time()

    def flush():
        for f in FRONTS:
            for n, mod in MODELS.items():
                t = time.time(); E[(n, f)].append(EMB[n](mod, buf[f])); secs[n] += time.time() - t
            buf[f].clear()

    for i, nm in enumerate(names):
        x, sr = read_all(zf.read(nm))
        assert sr == SR and x.shape[1] == 8, (nm, sr, x.shape)
        fe = front_ends(x, TAU[m][mid[i]])
        for f in FRONTS:
            buf[f].append(fe[f])
        if len(buf["CH0"]) == batch:
            flush()
        if i % 500 == 0:
            print(f"  {key}: {i}/{len(names)} clips · {time.time() - t0:.0f} s", flush=True)
    if buf["CH0"]:
        flush()
    meta = {"key": key, "zip_md5": LEVELS[key][2], "models": {n: INFO[n] for n in MODELS}, "model_seconds": {n: round(v, 1) for n, v in secs.items()},
            "runtime_s": round(time.time() - t0, 1), "device": DEVICE, "test_run": TEST, "date": time.strftime("%Y-%m-%d %H:%M")}
    np.savez_compressed(out, fname=names, mid=mid, label=label,
                        **{f"{n}_{f}": np.concatenate(v).astype(np.float32) for (n, f), v in E.items()}, meta=json.dumps(meta))
    print(f"{key}: saved {out.name} · {len(names)} clips · {meta['runtime_s']:.0f} s · model time {meta['model_seconds']}")
    if IN_COLAB and zpath.parent == WORK:
        zpath.unlink()                                                    # free the disk for the next zip


for key in LEVELS:
    run(key)

In [ ]:
# 7 · Scores (B1), fusion, gate G1, the selection on pumps, and decisions E1–E4 on fans
RES = {}
for key in LEVELS:
    Bk = BEATS[key]
    D = np.load(OUT / f"enc_{key.replace('|', '_')}.npz", allow_pickle=False)
    assert np.array_equal(D["fname"], Bk["fname"])
    mid, label, names = Bk["mid"], Bk["label"], Bk["fname"]
    for f in FRONTS:
        RES[(key, "BEATs", f)] = evaluate(Bk[f], mid, label, names)
        for n in MODELS:
            if f"{n}_{f}" not in D.files:
                continue
            En = D[f"{n}_{f}"].astype(np.float64)
            RES[(key, n, f)] = evaluate(En, mid, label, names)
            RES[(key, f"BEATs+{n}", f)] = evaluate(Bk[f], mid, label, names, fuse_with=En)
print(f"{'machine|level':12s} {'model':13s} {'front':5s} " + " ".join(f"{s:>7s}" for s in SPLITS))
for (key, n, f), r in RES.items():
    print(f"{key:12s} {n:13s} {f:5s} " + " ".join(f"{r[s]['avg']:7.4f}" for s in SPLITS))

G1 = {n: RES[("pump|6dB", n, "CH0")]["A"]["avg"] for n in MODELS if ("pump|6dB", n, "CH0") in RES}
ELIGIBLE = [n for n, v in G1.items() if v >= G1_MIN]
if not TEST:   # the fan bars use notebook 13's BEATs numbers; the Drive files must give exactly those
    assert RES[("fan|-6dB", "BEATs", "BF")]["A"]["avg"] == BEATS_FAN["A"] and RES[("fan|-6dB", "BEATs", "BF")]["B+10"]["avg"] == BEATS_FAN["B+10"], "stop: BEATs fan numbers differ from notebook 13"
print("\nG1 · pump CH0 Split A at 6 dB (≥ 0.80 to count):", G1, "→ eligible:", ELIGIBLE)
SEL_SCORE = {n: round((RES[("pump|-6dB", n, "BF")]["A"]["avg"] + RES[("pump|-6dB", n, "BF")]["B+10"]["avg"]) / 2, 4) for n in ELIGIBLE}
SEL = None
if ELIGIBLE:
    best = max(SEL_SCORE.values())
    tied = [n for n in ELIGIBLE if best - SEL_SCORE[n] <= 0.001]
    tied.sort(key=lambda n: (-RES[("pump|-6dB", n, "BF")]["A"]["avg"], n != "FISHER"))
    SEL = tied[0]
print("selection on pumps (−6 dB, BF, mean of A and B+10):", SEL_SCORE, "→ selected:", SEL,
      f"· BEATs for comparison {round((RES[('pump|-6dB', 'BEATs', 'BF')]['A']['avg'] + RES[('pump|-6dB', 'BEATs', 'BF')]['B+10']['avg']) / 2, 4)}")
DEC, VERDICT = {}, {}
if SEL:
    fa = RES[("fan|-6dB", SEL, "BF")]; fc = RES[("fan|-6dB", SEL, "CH0")]; ff = RES[("fan|-6dB", f"BEATs+{SEL}", "BF")]
    DEC = {"E1": (fa["A"]["avg"] >= BEATS_FAN["A"] + BAR, f"{SEL} BF A {fa['A']['avg']} vs BEATs {BEATS_FAN['A']} (bar {BEATS_FAN['A'] + BAR:.4f})"),
           "E2": (fa["B+10"]["avg"] >= BEATS_FAN["B+10"] + BAR, f"{SEL} BF B+10 {fa['B+10']['avg']} vs BEATs {BEATS_FAN['B+10']} (bar {BEATS_FAN['B+10'] + BAR:.4f})"),
           "E3": (fa["A"]["avg"] - fc["A"]["avg"] >= BAR_ARRAY, f"{SEL}: BF {fa['A']['avg']} − CH0 {fc['A']['avg']} = {fa['A']['avg'] - fc['A']['avg']:+.4f} (bar +0.01)"),
           "E4": (ff["A"]["avg"] >= BEATS_FAN["A"] + BAR, f"BEATs+{SEL} BF A {ff['A']['avg']} vs BEATs {BEATS_FAN['A']} (bar {BEATS_FAN['A'] + BAR:.4f})")}
    TXT = {"E1": ("a newer model helps a known fan in heavy noise", "no clear gain over BEATs for a known fan"),
           "E2": ("a newer model helps a new fan with 10 clips", "no clear gain over BEATs for a new fan"),
           "E3": ("the array still adds on top of the newer model", "the array adds nothing on top of the newer model"),
           "E4": ("BEATs and the newer model together help a known fan", "no clear gain from combining the two")}
    VERDICT = {k: TXT[k][0] if ok else TXT[k][1] for k, (ok, _) in DEC.items()}
else:
    VERDICT = {k: "not decidable: no new model passed G1" for k in ("E1", "E2", "E3", "E4")}
print()
for k in ("E1", "E2", "E3", "E4"):
    print(f"{k} → {VERDICT[k]}" + (f"  ·  {DEC[k][1]}" if k in DEC else ""))

In [ ]:
# 8 · Save, a picture, and one zip to download
RESULT = {"rules": RULES, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST, "models": INFO,
          "results": {f"{k}|{n}|{f}": r for (k, n, f), r in RES.items()},
          "G0": {k: {"got": v[0], "expected": v[1]} for k, v in G0.items()}, "G1": G1, "eligible": ELIGIBLE,
          "selection_score": SEL_SCORE, "selected": SEL,
          "decisions": {k: {"pass": bool(ok), "detail": d} for k, (ok, d) in DEC.items()}, "verdicts": VERDICT,
          "run_meta": {key: json.loads(str(np.load(OUT / f"enc_{key.replace('|', '_')}.npz")["meta"])) for key in LEVELS}}
name = "new_encoders.json" if not (OUT / "new_encoders.json").exists() else "new_encoders_rerun.json"
(OUT / name).write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))
import matplotlib.pyplot as plt
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
LINES = [("BEATs", "CH0", "#8d8d8d", "BEATs, one microphone"), ("BEATs", "BF", "#2a78d6", "BEATs, 8 aligned"),
         ("FISHER", "BF", "#eb6834", "FISHER, 8 aligned"), ("ECHO", "BF", "#1baf7a", "ECHO, 8 aligned")]
if SEL:
    LINES.append((f"BEATs+{SEL}", "BF", "#eda100", f"BEATs + {SEL}, 8 aligned"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True)
xs = np.arange(len(SPLITS))
for ax, m, title in ((axes[0], "pump", "pumps at −6 dB (used to choose the model)"), (axes[1], "fan", "fans at −6 dB (the decisions)")):
    for n, f, col, lab in LINES:
        if (f"{m}|-6dB", n, f) in RES:
            ax.plot(xs, [RES[(f"{m}|-6dB", n, f)][s]["avg"] for s in SPLITS], "-o", color=col, lw=2, ms=6, label=lab)
    ax.set_xticks(xs, ["known", "new, 0 clips", "1 clip", "3 clips", "10 clips"])
    ax.set_title(title, loc="left", color=INK, fontsize=10.5); ax.grid(True, color=GRID, lw=0.6); ax.tick_params(colors=MUTED)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[0].set_ylabel("average AUC", color=MUTED)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, loc="lower center", ncol=len(l), fontsize=9)
fig.tight_layout(rect=(0, 0.07, 1, 1)); fig.savefig(OUT / "new_encoders.png", dpi=150); plt.show()
ZIP = OUT / "p04_n15_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for n in (name, "new_encoders.png"):
        z.write(OUT / n, n)
print("saved", name, "· zipped →", ZIP)
if IN_COLAB:
    from google.colab import files as _f
    _f.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n15_results.zip`). If a cell fails, paste its error in the chat.